In [2]:
pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.7 MB/s eta 0:00:00


In [5]:
# ==================================================
# Assessment 2 – Translation with LLM (Colab ready)
# Fixed: get language token ID safely
# ==================================================

import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from sacrebleu import corpus_bleu, corpus_chrf, corpus_ter

# ------------------------------
# 0. Configuration
# ------------------------------
DATASET_PATH = "cleaned_dataset.xlsx"
N_SENTENCES = 100                 # number of sentences to translate

MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"             # English
TGT_LANG = "hin_Deva"             # Hindi

OUTPUT_EXCEL = "/content/cleaned_dataset.xlsx"
OUTPUT_SCORES = "scores.txt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ------------------------------
# 1. Load the cleaned dataset
# ------------------------------
df = pd.read_excel(DATASET_PATH)

assert "english" in df.columns, "Dataset must have 'english' column"
assert "hindi" in df.columns, "Dataset must have 'hindi' column with reference translations"

df = df.head(N_SENTENCES).reset_index(drop=True)
english_sentences = df["english"].tolist()
reference_hindi = df["hindi"].tolist()

print(f"Loaded {len(english_sentences)} sentences for translation.")

# ------------------------------
# 2. Load the translation model
# ------------------------------
print("Loading model (may download ~2.5 GB on first run)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# ------------------------------
# 3. Batch translation function
# ------------------------------
def translate_batch(sentences, batch_size=8):
    translations = []
    # Pre‑compute the Hindi language token ID
    forced_bos_token_id = tokenizer.encode(TGT_LANG, add_special_tokens=False)[0]

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_length=128,
            num_beams=5,
            early_stopping=True
        )
        batch_translations = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)
        translations.extend(batch_translations)

        print(f"Translated {min(i+batch_size, len(sentences))}/{len(sentences)}")
    return translations

# Run translation
print("Translating...")
model_translations = translate_batch(english_sentences, batch_size=8)  # reduce to 4 if CUDA OOM

# ------------------------------
# 4. Save translations to Excel
# ------------------------------
output_df = pd.DataFrame({
    "Original English sentence": english_sentences,
    "Model-generated Hindi translation": model_translations
})
output_df.to_excel(OUTPUT_EXCEL, index=False)
print(f"Translations saved to {OUTPUT_EXCEL}")

# ------------------------------
# 5. Compute BLEU, chrF, TER
# ------------------------------
refs = [[ref] for ref in reference_hindi]

bleu = corpus_bleu(model_translations, refs)
chrf = corpus_chrf(model_translations, refs)
ter  = corpus_ter(model_translations, refs)

with open(OUTPUT_SCORES, "w", encoding="utf-8") as f:
    f.write(f"Sentences evaluated: {len(english_sentences)}\n")
    f.write(f"Model: {MODEL_NAME}\n")
    f.write(f"BLEU score: {bleu.score:.2f}\n")
    f.write(f"chrF score: {chrf.score:.2f}\n")
    f.write(f"TER score:  {ter.score:.2f}\n")
    f.write("\nDetailed:\n")
    f.write(f"BLEU: {bleu}\n")
    f.write(f"chrF: {chrf}\n")
    f.write(f"TER:  {ter}\n")

print(f"\nScores saved to {OUTPUT_SCORES}")

# Print scores on screen
print("\n--- Evaluation Scores ---")
print(f"BLEU: {bleu.score:.2f}")
print(f"chrF: {chrf.score:.2f}")
print(f"TER:  {ter.score:.2f}")

# Optional: download files directly in Colab
# from google.colab import files
# files.download("translations.xlsx")
# files.download("scores.txt")

Using device: cuda
Loaded 100 sentences for translation.
Loading model (may download ~2.5 GB on first run)...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Translating...
Translated 8/100
Translated 16/100
Translated 24/100
Translated 32/100
Translated 40/100
Translated 48/100
Translated 56/100
Translated 64/100
Translated 72/100
Translated 80/100
Translated 88/100
Translated 96/100
Translated 100/100
Translations saved to /content/cleaned_dataset.xlsx

Scores saved to scores.txt

--- Evaluation Scores ---
BLEU: 22.00
chrF: 50.99
TER:  110.29
